# Six-Face IMU Calibration Capture

Use this notebook to collect six stationary captures, one for each face orientation, and compare the resulting histograms directly.

How to run the test:
1. Select the Python kernel from `signal/venv`.
2. Setup libraries and plotting.
3. Connect to the device.
4. For each orientation section: 

    4.1. Trigger notification streaming on the device.
    
    4.2. Place it still either on a flat surface or hanging.
    
    4.3. Trigger the capture cell.
    
    4.4. Run the plot cell.

5. Disconnect from the device and turn it off.

## Setup

In [1]:
from pathlib import Path
from statistics import fmean, pstdev

import matplotlib.pyplot as plt

from signal_utils import IMUSampleReceiver

LISTEN_DURATION_SECONDS = 10.0
G = 9.80665

ACCEL_RANGE = (-15.0, 15.0)
ACCEL_BIN_WIDTH = 0.25
GYRO_RANGE = (-20.0, 20.0)
GYRO_BIN_WIDTH = 0.25

OUTPUT_DIR = Path(".").resolve()

receiver = IMUSampleReceiver(listenDurationSeconds=LISTEN_DURATION_SECONDS)


def build_bin_edges(start: float, stop: float, width: float) -> list[float]:
    steps = int(round((stop - start) / width))
    return [start + index * width for index in range(steps + 1)]


ACCEL_BINS = build_bin_edges(ACCEL_RANGE[0], ACCEL_RANGE[1], ACCEL_BIN_WIDTH)
GYRO_BINS = build_bin_edges(GYRO_RANGE[0], GYRO_RANGE[1], GYRO_BIN_WIDTH)


def require_connected() -> None:
    if not receiver.isConnected():
        raise RuntimeError("Run the connect cell before sampling.")


async def sample_orientation(file_name: str):
    require_connected()
    file_path = OUTPUT_DIR / file_name
    print(f"Saving capture to {file_path}")
    return await receiver.receive(exportedFileName=str(file_path))


def format_reference(reference: float) -> str:
    if abs(reference - G) < 1e-9:
        return "+g"
    if abs(reference + G) < 1e-9:
        return "-g"
    return f"{reference:.3f}"


def plot_distribution(axis, values, bins, plot_range, reference, color, title, x_label):
    axis.set_title(title)
    axis.set_xlabel(x_label)
    axis.set_ylabel("count")
    axis.set_xlim(*plot_range)
    axis.grid(alpha=0.25)

    if len(values) == 0:
        axis.text(
            0.5, 0.5, "No samples", transform=axis.transAxes, ha="center", va="center"
        )
        return

    mean_value = fmean(values)
    std_value = pstdev(values) if len(values) > 1 else 0.0

    axis.hist(values, bins=bins, color=color, alpha=0.75, edgecolor="white")
    axis.axvline(
        reference,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"reference = {format_reference(reference)}",
    )
    axis.axvline(
        mean_value,
        color="crimson",
        linestyle="-",
        linewidth=1.5,
        label=f"mean = {mean_value:.3f}",
    )
    axis.plot([], [], linestyle="", label=f"std = {std_value:.3f}")
    axis.legend(loc="upper right")


def plot_orientation_histograms(
    motion_data, title: str, accel_references: dict[str, float]
) -> None:
    if len(motion_data["t"]) == 0:
        raise ValueError("No samples available for plotting.")

    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    accel_plots = [
        (
            axes[0, 0],
            motion_data["a"]["x"],
            accel_references["x"],
            "tab:blue",
            "Accel X",
            "m/s^2",
        ),
        (
            axes[1, 0],
            motion_data["a"]["y"],
            accel_references["y"],
            "tab:green",
            "Accel Y",
            "m/s^2",
        ),
        (
            axes[2, 0],
            motion_data["a"]["z"],
            accel_references["z"],
            "tab:orange",
            "Accel Z",
            "m/s^2",
        ),
    ]
    gyro_plots = [
        (axes[0, 1], motion_data["w"]["roll"], 0.0, "tab:purple", "Gyro Roll", "deg/s"),
        (axes[1, 1], motion_data["w"]["pitch"], 0.0, "tab:red", "Gyro Pitch", "deg/s"),
        (axes[2, 1], motion_data["w"]["yaw"], 0.0, "tab:brown", "Gyro Yaw", "deg/s"),
    ]

    for axis, values, reference, color, subplot_title, x_label in accel_plots:
        plot_distribution(
            axis,
            values,
            ACCEL_BINS,
            ACCEL_RANGE,
            reference,
            color,
            subplot_title,
            x_label,
        )

    for axis, values, reference, color, subplot_title, x_label in gyro_plots:
        plot_distribution(
            axis,
            values,
            GYRO_BINS,
            GYRO_RANGE,
            reference,
            color,
            subplot_title,
            x_label,
        )

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


print(f"CSV output directory: {OUTPUT_DIR}")

CSV output directory: /home/fedep/Repos/gym-imu/signal/calibration


## Connect

Run the next cell once before starting the six captures.

In [ ]:
if receiver.isConnected():
    print("Receiver already connected")
else:
    await receiver.connect()

## Orientation 1: +X Up

<img src="./orientation-reference/positive-x-up.jpg" alt="Positive X Up" width="250" height="250">


In [ ]:
x_positive_data = await sample_orientation("positive-x-up.csv")

In [ ]:
plot_orientation_histograms(
    x_positive_data,
    title="Six-Face Test: +X Up",
    accel_references={"x": G, "y": 0.0, "z": 0.0},
)

## Orientation 2: -X Up

<img src="./orientation-reference/negative-x-up.jpg" alt="Negative X Up" width="250" height="250">

In [ ]:
x_negative_data = await sample_orientation("negative-x-up.csv")

In [ ]:
plot_orientation_histograms(
    x_negative_data,
    title="Six-Face Test: -X Up",
    accel_references={"x": -G, "y": 0.0, "z": 0.0},
)

## Orientation 3: +Y Up

<img src="./orientation-reference/positive-y-up.jpg" alt="Positive Y Up" width="250" height="250">

In [ ]:
y_positive_data = await sample_orientation("positive-y-up.csv")

In [ ]:
plot_orientation_histograms(
    y_positive_data,
    title="Six-Face Test: +Y Up",
    accel_references={"x": 0.0, "y": G, "z": 0.0},
)

## Orientation 4: -Y Up

<img src="./orientation-reference/negative-y-up.jpg" alt="Negative Y Up" width="250" height="250">

In [ ]:
y_negative_data = await sample_orientation("negative-y-up.csv")

In [ ]:
plot_orientation_histograms(
    y_negative_data,
    title="Six-Face Test: -Y Up",
    accel_references={"x": 0.0, "y": -G, "z": 0.0},
)

## Orientation 5: +Z Up

<img src="./orientation-reference/positive-z-up.jpg" alt="Positive Z Up" width="250" height="250">

In [ ]:
z_positive_data = await sample_orientation("positive-z-up.csv")

In [ ]:
plot_orientation_histograms(
    z_positive_data,
    title="Six-Face Test: +Z Up",
    accel_references={"x": 0.0, "y": 0.0, "z": G},
)

## Orientation 6: -Z Up

<img src="./orientation-reference/negative-z-up.jpg" alt="Negative Z Up" width="250" height="250">

In [ ]:
z_negative_data = await sample_orientation("negative-z-up.csv")

In [ ]:
plot_orientation_histograms(
    z_negative_data,
    title="Six-Face Test: -Z Up",
    accel_references={"x": 0.0, "y": 0.0, "z": -G},
)

## Disconnect

Run the next cell after the sixth plot finishes.

In [ ]:
await receiver.disconnect()